In [183]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jgassdfe/pokemon-dataset-of-gen-1-gen-9/pokemon_data.csv
/kaggle/input/datasets/adampq/pokemon-tcg-all-cards-1999-2023/pokemon-tcg-data-master 1999-2023.csv
/kaggle/input/datasets/adampq/pokemon-tcg-all-cards-1999-2023/pokemon-tcg-data-master 1999-2023 Data Dictionary.txt


<h3>Importations

In [184]:
import pandas as pd
import numpy as np
!pip install ctgan
from ctgan import CTGAN
import ast
import re

<h3>Lecture de données</h3>

In [185]:
df_poke = pd.read_csv("/kaggle/input/datasets/jgassdfe/pokemon-dataset-of-gen-1-gen-9/pokemon_data.csv")
df_poke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 48 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      1025 non-null   int64  
 1   Name                    1025 non-null   object 
 2   HP                      1025 non-null   int64  
 3   Attack                  1025 non-null   int64  
 4   Defense                 1025 non-null   int64  
 5   Sp. Attack              1025 non-null   int64  
 6   Sp. Defense             1025 non-null   int64  
 7   Speed                   1025 non-null   int64  
 8   Base_Stats              1025 non-null   int64  
 9   normal_weakness         1025 non-null   float64
 10  fire_weakness           1025 non-null   float64
 11  water_weakness          1025 non-null   float64
 12  electric_weakness       1025 non-null   float64
 13  grass_weakness          1025 non-null   float64
 14  ice_weakness            1025 non-null   

In [186]:
df_cards = pd.read_csv("/kaggle/input/datasets/adampq/pokemon-tcg-all-cards-1999-2023/pokemon-tcg-data-master 1999-2023.csv")
df_cards.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17172 entries, 0 to 17171
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      17172 non-null  object 
 1   set                     17172 non-null  object 
 2   series                  17172 non-null  object 
 3   publisher               17172 non-null  object 
 4   generation              17172 non-null  object 
 5   release_date            17172 non-null  object 
 6   artist                  16951 non-null  object 
 7   name                    17172 non-null  object 
 8   set_num                 17172 non-null  object 
 9   types                   14500 non-null  object 
 10  supertype               17172 non-null  object 
 11  subtypes                16998 non-null  object 
 12  level                   2404 non-null   object 
 13  hp                      14536 non-null  float64
 14  evolvesFrom             6165 non-null 

In [187]:
cols = [
    "HP", "Attack", "Defense", "Sp. Attack",
    "Sp. Defense", "Speed",
    "Type 1", "Type 2",
    "Is_Legendary"
]

df_poke = df_poke[cols].copy()

df_poke["Type 2"] = df_poke["Type 2"].fillna("None")

df_poke.head()

,HP,Attack,Defense,Sp. Attack,Sp. Defense,Speed,Type 1,Type 2,Is_Legendary
0,45,49,49,65,65,45,Grass,Poison,0
1,60,62,63,80,80,60,Grass,Poison,0
2,80,82,83,100,100,80,Grass,Poison,0
3,39,52,43,60,50,65,Fire,None,0
4,58,64,58,80,65,80,Fire,None,0


In [188]:
def parse_attacks(x):
    try:
        return ast.literal_eval(x)
    except:
        return None

In [189]:
df_attacks = df_cards.dropna(subset=["attacks"]).copy()
df_attacks["attacks"] = df_attacks["attacks"].apply(parse_attacks)
df_attacks = df_attacks.explode("attacks")
df_attacks = df_attacks.reset_index(drop=True)
df_attacks = df_attacks.dropna(subset=["attacks"])
df_attacks.head()

,id,set,series,publisher,generation,release_date,artist,name,set_num,types,...,retreatCost,convertedRetreatCost,rarity,flavorText,nationalPokedexNumbers,legalities,resistances,rules,regulationMark,ancientTrait
0,base1-1,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Alakazam,1,['Psychic'],...,"['Colorless', 'Colorless', 'Colorless']",3.0,Rare Holo,Its brain can outperform a supercomputer. Its ...,[65],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN
1,base1-2,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Blastoise,2,['Water'],...,"['Colorless', 'Colorless', 'Colorless']",3.0,Rare Holo,A brutal Pokémon with pressurized water jets o...,[9],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN
2,base1-3,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Chansey,3,['Colorless'],...,['Colorless'],1.0,Rare Holo,A rare and elusive Pokémon that is said to bri...,[113],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN
3,base1-3,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Chansey,3,['Colorless'],...,['Colorless'],1.0,Rare Holo,A rare and elusive Pokémon that is said to bri...,[113],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN
4,base1-4,Base,Base,WOTC,First,1/9/1999,Mitsuhiro Arita,Charizard,4,['Fire'],...,"['Colorless', 'Colorless', 'Colorless']",3.0,Rare Holo,Spits fire that is hot enough to melt boulders...,[6],{'unlimited': 'Legal'},"[{'type': 'Fighting', 'value': '-30'}]",NaN,NaN,NaN


<h3>CTGAN</h3>

In [190]:
categorical_cols = ["Type 1", "Type 2", "Is_Legendary"]

In [191]:
ctgan = CTGAN(
    epochs=300,
    batch_size=64,
    pac = 8,
    verbose=True
)

In [192]:
ctgan.fit(df_poke, categorical_cols)

Gen. (-01.96) | Discrim. (-00.09): 100%|██████████| 300/300 [02:05<00:00,  2.39it/s]


<h3>Génération</h3> 

In [193]:
sample = ctgan.sample(1)
sample

,HP,Attack,Defense,Sp. Attack,Sp. Defense,Speed,Type 1,Type 2,Is_Legendary
0,81,79,69,67,59,59,Rock,Ghost,0


<h3>Nettoyage</h3>

In [194]:
def clean_stats(row):
    for col in ["HP", "Attack", "Defense", "Sp. Attack", "Sp. Defense", "Speed"]:
        row[col] = int(np.clip(row[col], 10, 255))
    return row

sample = sample.apply(clean_stats, axis=1)
sample

,HP,Attack,Defense,Sp. Attack,Sp. Defense,Speed,Type 1,Type 2,Is_Legendary
0,81,79,69,67,59,59,Rock,Ghost,0


<h3>Conversion pour cartes Pokémon

In [195]:
def convert_to_card(pokemon):
    hp_game = pokemon["HP"]
    
    hp_card = int(hp_game * 1.5)
    hp_card = int(np.clip(hp_card, 40, 180))
    
    retreat = int((pokemon["Defense"] + pokemon["Sp. Defense"]) / 100)
    retreat = int(np.clip(retreat, 0, 4))
    
    return {
        "HP": hp_card,
        "Type": pokemon["Type 1"],
        "Retreat": retreat,
        "Legendary": bool(pokemon["Is_Legendary"])
    }

card_stats = convert_to_card(sample.iloc[0])
card_stats

{'HP': 121, 'Type': 'Rock', 'Retreat': 1, 'Legendary': False}

<h3>Structuration des attaques</h3>

In [196]:
def extract_attack_info(a):
    return {
        "name": a.get("name"),
        "damage": a.get("damage"),
        "text": a.get("text"),
        "cost": a.get("cost")
    }

df_attacks = df_cards.dropna(subset=["attacks"]).copy()

df_attacks["attacks"] = df_attacks["attacks"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_attacks = df_attacks.explode("attacks").reset_index(drop=True)


df_attacks_struct.head()

,name,damage,text,cost,type
0,Confuse Ray,30,"Flip a coin. If heads, the Defending Pokémon i...","[Psychic, Psychic, Psychic]",['Psychic']
1,Hydro Pump,40+,Does 40 damage plus 10 more damage for each Wa...,"[Water, Water, Water]",['Water']
2,Scrunch,,"Flip a coin. If heads, prevent all damage done...","[Colorless, Colorless]",['Colorless']
3,Double-edge,80,Chansey does 80 damage to itself.,"[Colorless, Colorless, Colorless, Colorless]",['Colorless']
4,Fire Spin,100,Discard 2 Energy cards attached to Charizard i...,"[Fire, Fire, Fire, Fire]",['Fire']


In [197]:
df_attacks_struct = pd.DataFrame(
    df_attacks["attacks"].apply(extract_attack_info).tolist()
)

df_attacks_struct["type"] = df_attacks["types"].reset_index(drop=True)

<h3>Nettoyage</h3>

In [198]:
def parse_damage(dmg):
    if dmg is None:
        return 0
    match = re.search(r"\d+", str(dmg))
    return int(match.group()) if match else 0

df_attacks_struct["damage_num"] = df_attacks_struct["damage"].apply(parse_damage)

<h3>Type</h3>

In [199]:
df_attacks_struct["type"] = df_attacks["types"].reset_index(drop=True)
df_attacks_struct = pd.DataFrame(
    df_attacks["attacks"].apply(extract_attack_info).tolist()
)

df_attacks_struct["type"] = df_attacks["types"]

df_attacks_struct.head()

,name,damage,text,cost,type
0,Confuse Ray,30,"Flip a coin. If heads, the Defending Pokémon i...","[Psychic, Psychic, Psychic]",['Psychic']
1,Hydro Pump,40+,Does 40 damage plus 10 more damage for each Wa...,"[Water, Water, Water]",['Water']
2,Scrunch,,"Flip a coin. If heads, prevent all damage done...","[Colorless, Colorless]",['Colorless']
3,Double-edge,80,Chansey does 80 damage to itself.,"[Colorless, Colorless, Colorless, Colorless]",['Colorless']
4,Fire Spin,100,Discard 2 Energy cards attached to Charizard i...,"[Fire, Fire, Fire, Fire]",['Fire']


<h3>Génération d'attaque</h3>

In [200]:
def generate_attack(pokemon, df_attacks):
    atk = generate_attack_from_data(pokemon, df_attacks)
    
    atk["damage"] = scale_damage(atk["damage"], pokemon)
    
    atk["energy"] = max(1, atk["damage"] // 50)
    
    return atk

In [201]:
def generate_attack_from_data(pokemon, df_attacks):
    poke_type = pokemon["Type 1"]
    
    subset = df_attacks[
        df_attacks["type"].astype(str).str.contains(poke_type, na=False)
    ]
    
    if len(subset) == 0:
        subset = df_attacks
    
    attack = subset.sample(1).iloc[0]
    
    base_damage = parse_damage(attack["damage"])
    
    return {
        "name": attack["name"],
        "damage": base_damage,
        "effect": attack["text"] if attack["text"] else "No effect",
        "cost": attack["cost"]
    }

In [202]:
def scale_damage(base_damage, pokemon):
    atk = pokemon["Attack"]
    sp_atk = pokemon["Sp. Attack"]
    
    scale = (atk + sp_atk) / 150
    dmg = int(base_damage * scale)
    
    return max(10, min(dmg, 200))

In [203]:
def generate_attack(pokemon, df_attacks):
    atk = generate_attack_from_data(pokemon, df_attacks)
    
    atk["damage"] = scale_damage(atk["damage"], pokemon)
    atk["energy"] = max(1, atk["damage"] // 50)
    
    return atk

In [206]:
attacks = generate_attacks(sample.iloc[0], df_attacks_struct)
attacks

[{'name': 'Chemical Breath',
  'damage': 10,
  'effect': "This attack does 70 more damage for each Special Condition affecting your opponent's Active Pokémon.",
  'cost': ['Darkness', 'Colorless', 'Colorless'],
  'energy': 1}]

In [207]:
for atk in attacks:
    print(f"{atk['name']}")
    print(f"Damage: {atk['damage']}")
    print(f"Energy: {atk['energy']}")
    print(f"Effect: {atk['effect']}")
    print("-"*30)

Chemical Breath
Damage: 10
Energy: 1
Effect: This attack does 70 more damage for each Special Condition affecting your opponent's Active Pokémon.
------------------------------
